# PST-Parser - parse dei 600 task con il parser r4 (analisi esplorativa R)

Derivato da `notebooks/hauto-parser.ipynb` (deviazione del 22/11/2026). Gia fissati: `EXPECTED_TREE` (72b7e9d0ae83…), `RUN_ID` `hauto-r4-01`, `PURPOSE` pilot, adapter della run `parser-2026-11-22-r4` su Drive, identita `results/r4/parser-r4.json`. Stessa configurazione e decodifica del parse di r3 (`configs/experiments/hauto-parser.yaml`, greedy, NVIDIA L4). File dei prompt da caricare: lo stesso del parse di r3 (`sources.jsonl`, 600 task, SHA `76b12e23…`).

Lanciatore del parser di H-auto (M19): **nessuna logica propria**. Ogni cella chiama i comandi del progetto e si ferma con un errore se un controllo fallisce.

Il parser e l'adapter della run `parser-2026-11-12-r3` sui pesi su cui e stato addestrato, `unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit@0db785ab...`, caricati con il backend `hf` (configurazione `configs/experiments/hauto-parser.yaml`). `parse-prompts --purpose pilot|final` verifica, prima di caricare, modello, revisione, digest dei file dello snapshot, adapter, prompt di sistema e decoding contro `protocol/hauto/parser.json`, e dopo il caricamento il template della conversazione.

Che cosa serve:
1. l'archivio sorgente creato in locale con `git -c core.autocrlf=false archive --format=zip -o pst-src.zip HEAD` e il tree stampato da `git rev-parse "HEAD^{tree}"`;
2. il file dei prompt da analizzare (JSONL con `task_id`, `text` e, se ci sono, gli span del gold): per il pilota D-01 `pilot-parse-sources.jsonl`;
3. l'adapter della run di riferimento su Drive (cartella `ADAPTER` della cella dell'identita).

Scopi: `pilot` (D-01, nessun freeze; i digest del modello si producono qui con `model-digests`) e `final` (gli 800 task di S_gold e gli altri di S_full, con il freeze H-auto; i digest sono quelli congelati in `protocol/hauto/parser-model.json`).

**Ripresa.** Dopo un'interruzione: runtime nuovo, stesso `pst-src.zip`, stesse impostazioni della cella dell'identita, poi rieseguire tutte le celle. `parse-prompts --resume` continua dopo l'ultima predizione completa e rifiuta la ripresa se il piano (sorgenti, configurazione, adapter, identita, digest, freeze) e cambiato.


In [ ]:
import subprocess

# GPU dichiarata per pilot, nucleo, determinismo e H-auto (00 G-1): NVIDIA L4.
DECLARED_GPU = "NVIDIA L4"
found = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(found)
if not found or any(line.split(",")[0].strip() != DECLARED_GPU for line in found.splitlines()):
    raise RuntimeError(f"Serve un runtime con {DECLARED_GPU}")


In [ ]:
import hashlib
import os
import re
import sys
import zipfile
from pathlib import Path

from google.colab import files

EXPECTED_TREE = "72b7e9d0ae8345954c034dfe70f948eb9e4d5ef8"  # git rev-parse "HEAD^{tree}" in locale
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_TREE):
    raise ValueError("Impostare il tree del commit da eseguire")

uploaded = files.upload()  # selezionare solo pst-src.zip
if len(uploaded) != 1:
    raise ValueError("Selezionare un solo archivio sorgente ZIP")
archive_name, archive_bytes = next(iter(uploaded.items()))
SOURCE_ARCHIVE = Path("/content") / archive_name
SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
PROJECT = Path("/content/pst-src")
if PROJECT.exists():
    raise FileExistsError("Sorgente gia estratta: usare un runtime nuovo")
PROJECT.mkdir()
with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
    for member in bundle.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError("Archivio con un percorso esterno alla cartella sorgente")
        if Path(member.filename).name == ".env":
            raise ValueError("Rimuovere .env dall'archivio sorgente")
    bundle.extractall(PROJECT)
if not (PROJECT / "pyproject.toml").is_file():
    raise ValueError("L'archivio deve avere il progetto alla radice (git archive HEAD)")
# Fine riga LF, hash dei file chiave e tree ricalcolato dal contenuto.
subprocess.run(
    [
        sys.executable,
        str(PROJECT / "scripts" / "verify_archive.py"),
        str(SOURCE_ARCHIVE),
        "--expected-tree",
        EXPECTED_TREE,
    ],
    check=True,
)
os.chdir(PROJECT)
print("source archive sha256:", SOURCE_ARCHIVE_SHA256)


In [ ]:
# Backend hf: bastano gli extra cu128 e train (transformers, peft, bitsandbytes).
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.15"], check=True)
EXTRAS = ["--extra", "cu128", "--extra", "train"]
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11", *EXTRAS], check=True)
UV_RUN = ["uv", "run", "--frozen", *EXTRAS]
PSTPARSER = [*UV_RUN, "pstparser"]
EMPIRICAL = [*UV_RUN, "python", "-m", "pstparser.empirical"]
subprocess.run([*UV_RUN, "python", "-c", "import pstparser, torch, transformers, peft"], check=True)


In [ ]:
import json

from google.colab import drive

drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"

RUN_ID = "hauto-r4-01"  # hauto-<fase>-NN; cambiare per ogni run NUOVA
PURPOSE = "pilot"  # "pilot": D-01 e parse del prefisso, prima del freeze unico F4; "final" solo per un nuovo parse dopo F4
FREEZE_SHA256 = None  # obbligatorio per "final": SHA-256 di protocol/protocol-freeze.json (F4)
ADAPTER = Path("/content/drive/MyDrive/pst-parser-runs/parser-2026-11-22-r4/training/adapter")
CONFIG = PROJECT / "configs" / "experiments" / "hauto-parser.yaml"
PARSER_IDENTITY = PROJECT / "results" / "r4" / "parser-r4.json"

if not re.fullmatch(r"hauto-[a-z0-9]+-[0-9]{2}", RUN_ID):
    raise ValueError("RUN_ID deve avere la forma hauto-<fase>-NN")
if PURPOSE not in {"pilot", "final"}:
    raise ValueError("PURPOSE deve essere pilot o final")
if (PURPOSE == "final") != (FREEZE_SHA256 is not None):
    raise ValueError("Una run final richiede FREEZE_SHA256; una run pilot non lo usa")
if not ADAPTER.is_dir():
    raise FileNotFoundError(f"Adapter assente: {ADAPTER}")

RUN_ROOT = Path("/content/drive/MyDrive/pst-hauto-runs") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_IDENTITY = RUN_ROOT / "run-identity.json"
SOURCES = RUN_ROOT / "sources.jsonl"
if not SOURCES.exists():
    if RUN_IDENTITY.exists():
        raise FileNotFoundError("Identita registrata ma sources.jsonl assente: non riprendere")
    uploaded = files.upload()  # selezionare solo il file dei prompt (JSONL)
    if len(uploaded) != 1:
        raise ValueError("Selezionare un solo file di prompt")
    SOURCES.write_bytes(next(iter(uploaded.values())))
SOURCES_SHA256 = hashlib.sha256(SOURCES.read_bytes()).hexdigest()
model = json.loads(PARSER_IDENTITY.read_text(encoding="utf-8"))["model"]

identity = {
    "run_id": RUN_ID,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "source_tree": EXPECTED_TREE,
    "freeze_sha256": FREEZE_SHA256,
    "purpose": PURPOSE,
    "sources_sha256": SOURCES_SHA256,
    "parser_identity_sha256": hashlib.sha256(PARSER_IDENTITY.read_bytes()).hexdigest(),
    "model_name": model["name"],
    "model_revision": model["revision"],
    "adapter": str(ADAPTER),
}
if RUN_IDENTITY.exists():
    if json.loads(RUN_IDENTITY.read_text(encoding="utf-8")) != identity:
        raise ValueError("RUN_ID gia associata a un'altra identita: usare una RUN_ID nuova")
    print("Ripresa della run", RUN_ID)
else:
    RUN_IDENTITY.write_text(json.dumps(identity, indent=2) + "\n", encoding="utf-8")
print(json.dumps(identity, indent=2))


In [ ]:
# Digest attesi dei file dello snapshot del modello base, pesi compresi.
if PURPOSE == "final":
    # Comando freeze-verify del progetto: il freeze unico F4 deve combaciare con la sorgente.
    FREEZE = PROJECT / "protocol" / "protocol-freeze.json"
    if hashlib.sha256(FREEZE.read_bytes()).hexdigest() != FREEZE_SHA256:
        raise ValueError("protocol-freeze.json diverso da FREEZE_SHA256")
    subprocess.run([*EMPIRICAL, "freeze-verify", str(FREEZE)], check=True)
    PARSER_MODEL = PROJECT / "protocol" / "hauto" / "parser-model.json"
else:
    # Pilot: model-digests scarica lo snapshot alla revisione fissata e ne calcola i digest.
    PARSER_MODEL = RUN_ROOT / "parser-model.json"
    if not PARSER_MODEL.exists():
        subprocess.run(
            [
                *EMPIRICAL,
                "model-digests",
                "--name",
                model["name"],
                "--revision",
                model["revision"],
                "--out",
                str(PARSER_MODEL),
            ],
            check=True,
        )
print("parser-model.json sha256:", hashlib.sha256(PARSER_MODEL.read_bytes()).hexdigest())


## Parser sui prompt

Una sola esecuzione per task, prima di ogni generazione H-auto. Le predizioni restano su Drive riga per riga; `score` e `auto-annotate` si eseguono in locale sui file riportati.


In [ ]:
PARSE_DIR = RUN_ROOT / "parse"
arguments = [
    *PSTPARSER,
    "parse-prompts",
    "--config",
    str(CONFIG),
    "--sources",
    str(SOURCES),
    "--run-dir",
    str(PARSE_DIR),
    "--adapter",
    str(ADAPTER),
    "--purpose",
    PURPOSE,
    "--identity",
    str(PARSER_IDENTITY),
    "--parser-model",
    str(PARSER_MODEL),
]
if PURPOSE == "final":
    arguments += ["--freeze", str(FREEZE)]
if (PARSE_DIR / "parse-prompts_manifest.json").exists():
    print("Parsing gia completato per questa RUN_ID")
else:
    if (PARSE_DIR / "parse-plan.json").exists():
        arguments.append("--resume")
    subprocess.run(arguments, check=True)


In [ ]:
# Archivio di trasporto: identita, digest del modello, sorgenti, predizioni e manifest.
number = 1
while (RUN_ROOT / f"{RUN_ID}-transport-{number:02d}.zip").exists():
    number += 1
TRANSPORT = RUN_ROOT / f"{RUN_ID}-transport-{number:02d}.zip"
with zipfile.ZipFile(TRANSPORT, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(RUN_IDENTITY, "run-identity.json")
    bundle.write(PARSER_MODEL, "parser-model.json")
    bundle.write(SOURCES, "sources.jsonl")
    for path in sorted(PARSE_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, f"parse/{path.relative_to(PARSE_DIR).as_posix()}")
print(TRANSPORT, hashlib.sha256(TRANSPORT.read_bytes()).hexdigest())
files.download(str(TRANSPORT))
